In [31]:
!pip install -q transformers accelerate

In [32]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
tok = AutoTokenizer.from_pretrained(MODEL)
tok.pad_token = tok.eos_token
tok.padding_side = "left"
model = AutoModelForCausalLM.from_pretrained(
    MODEL, torch_dtype=torch.float16, device_map="cuda")

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

In [33]:
import time, threading
from transformers import TextIteratorStreamer

def prompt_of_len(n_tokens: int) -> str:
    base = "Explain the following in detail.\n"
    filler = "A data center serves many inference requests at once. " * 600
    ids = tok(base + filler)["input_ids"][:n_tokens]
    return tok.decode(ids)

def measure_stream(prompt: str, new_tokens: int = 128):
    enc = tok(prompt, return_tensors="pt").to("cuda")
    streamer = TextIteratorStreamer(tok, skip_prompt=True,
                                    skip_special_tokens=True)
    kwargs = dict(**enc, max_new_tokens=new_tokens, do_sample=False,
                  streamer=streamer)
    th = threading.Thread(target=model.generate, kwargs=kwargs)
    t0 = time.time()
    th.start()
    stamps = []
    for _ in streamer:
        stamps.append(time.time())
    th.join()
    ttft = stamps[0] - t0
    if len(stamps) > 1:
        gaps = [b - a for a, b in zip(stamps, stamps[1:])]
        tpot = sum(gaps) / len(gaps)
    else:
        tpot = 0.0
    total = stamps[-1] - t0
    return {"ttft_s": round(ttft, 4), "tpot_s": round(tpot, 4),
            "total_s": round(total, 4), "n_tokens": len(stamps)}

measure_stream(prompt_of_len(128), new_tokens=8)  # warm-up، رميها

ttft_by_len = {}
for n in [128, 512, 2048]:
    r = measure_stream(prompt_of_len(n))
    ttft_by_len[str(n)] = r["ttft_s"]
    print(n, r)

128 {'ttft_s': 0.1757, 'tpot_s': 0.0782, 'total_s': 10.1877, 'n_tokens': 129}
512 {'ttft_s': 0.1478, 'tpot_s': 0.0423, 'total_s': 5.5562, 'n_tokens': 129}
2048 {'ttft_s': 0.6799, 'tpot_s': 0.0391, 'total_s': 5.6911, 'n_tokens': 129}


In [34]:
def cache_bytes(pkv):
    tensors = []
    if hasattr(pkv, "layers"):
        for layer in pkv.layers:
            for t in (getattr(layer, "keys", None), getattr(layer, "values", None)):
                if t is not None:
                    tensors.append(t)
    elif hasattr(pkv, "key_cache"):
        for t in list(pkv.key_cache) + list(pkv.value_cache):
            if t is not None:
                tensors.append(t)
    else:
        for layer in pkv:
            for t in layer:
                if t is not None:
                    tensors.append(t)
    return sum(t.numel() * t.element_size() for t in tensors)

In [35]:
import inspect
print(inspect.getsource(cache_bytes))

def cache_bytes(pkv):
    tensors = []
    if hasattr(pkv, "layers"):
        for layer in pkv.layers:
            for t in (getattr(layer, "keys", None), getattr(layer, "values", None)):
                if t is not None:
                    tensors.append(t)
    elif hasattr(pkv, "key_cache"):
        for t in list(pkv.key_cache) + list(pkv.value_cache):
            if t is not None:
                tensors.append(t)
    else:
        for layer in pkv:
            for t in layer:
                if t is not None:
                    tensors.append(t)
    return sum(t.numel() * t.element_size() for t in tensors)



In [44]:
formula = kv_formula_kb_per_token()
print("formula KB/token:", formula)

kv_rows = [measure_kv(c) for c in [512, 2048, 4096]]
for r in kv_rows:
    print(r, "  vs formula", formula, "KB/token")

import json, os
with open("kv_check.json", "w") as f:
    json.dump({"formula_kb_per_token": formula,
               "measured_kb_per_token": kv_rows[-1]["kv_kb_per_token"],
               "peak_kb_per_token": kv_rows[-1]["peak_kb_per_token"]}, f)

print("kv_check.json ", os.path.exists("kv_check.json"))

formula KB/token: 28.0
{'context': 512, 'total_tokens': 768, 'peak_kb_per_token': 82.9, 'kv_kb_per_token': 28.0}   vs formula 28.0 KB/token
{'context': 2048, 'total_tokens': 2304, 'peak_kb_per_token': 258.4, 'kv_kb_per_token': 28.0}   vs formula 28.0 KB/token
{'context': 4096, 'total_tokens': 4352, 'peak_kb_per_token': 484.5, 'kv_kb_per_token': 28.0}   vs formula 28.0 KB/token
kv_check.json  True


In [46]:
import os
print(os.path.exists("kv_check.json"))

True


In [45]:
print("kv_rows:", kv_rows)

kv_rows: [{'context': 512, 'total_tokens': 768, 'peak_kb_per_token': 82.9, 'kv_kb_per_token': 28.0}, {'context': 2048, 'total_tokens': 2304, 'peak_kb_per_token': 258.4, 'kv_kb_per_token': 28.0}, {'context': 4096, 'total_tokens': 4352, 'peak_kb_per_token': 484.5, 'kv_kb_per_token': 28.0}]


In [38]:
QUEUE = [32, 32, 32, 256] * 6

def static_queue(batch: int, prompt: str = "Explain what an inference server does."):
    t0 = time.time(); useful = 0; slots = 0
    for i in range(0, len(QUEUE), batch):
        chunk = QUEUE[i:i + batch]
        n = max(chunk)
        enc = tok([prompt] * len(chunk), return_tensors="pt",
                  padding=True).to("cuda")
        model.generate(**enc, max_new_tokens=n, do_sample=False)
        useful += sum(chunk)
        slots += n * len(chunk)
    dt = time.time() - t0
    return {"batch": batch, "wall_s": round(dt, 2),
            "tokens_per_s": round(useful / dt, 1),
            "slot_efficiency": round(useful / slots, 3)}

batch_rows = {}
for n in [1, 4, 8]:
    r = static_queue(n)
    batch_rows[str(n)] = r["tokens_per_s"]
    print(r)

{'batch': 1, 'wall_s': 78.84, 'tokens_per_s': 26.8, 'slot_efficiency': 1.0}
{'batch': 4, 'wall_s': 48.63, 'tokens_per_s': 43.4, 'slot_efficiency': 0.344}
{'batch': 8, 'wall_s': 24.87, 'tokens_per_s': 84.9, 'slot_efficiency': 0.344}


In [39]:
import json
baselines = {
    "model": MODEL,
    "dtype": "fp16",
    "ttft_s": ttft_by_len,
    "tpot_s": measure_stream(prompt_of_len(512))["tpot_s"],
    "batch": {k: v for k, v in batch_rows.items()},
}
with open("baselines.json", "w") as f:
    json.dump(baselines, f, indent=2)
print(json.dumps(baselines, indent=2))

{
  "model": "Qwen/Qwen2.5-1.5B-Instruct",
  "dtype": "fp16",
  "ttft_s": {
    "128": 0.1757,
    "512": 0.1478,
    "2048": 0.6799
  },
  "tpot_s": 0.044,
  "batch": {
    "1": 26.8,
    "4": 43.4,
    "8": 84.9
  }
}


In [40]:
from google.colab import files
files.download("baselines.json")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [47]:
# Green-check verifier for Lab W3D2 (inference anatomy).
# Paste this as the last cell of your day-2 notebook and run it. It reads
# baselines.json (the export you also downloaded) plus the KV measurement you
# saved to kv_check.json, and checks the schema and the sanity rules.
#
# Last line is exactly one of:
#   GREEN CHECK: PASS
#   GREEN CHECK: FAIL (<reason>)
# No interactivity, no arguments; exit code matches.

import json, os

# Qwen2.5-1.5B KV cache: 2 x 28 layers x 2 kv_heads x 128 head_dim x 2 bytes.
KV_FORMULA_KB_PER_TOKEN = 2 * 28 * 2 * 128 * 2 / 1024  # 28.0


class _Stop(Exception):
    """Ends the check without killing the notebook kernel."""


def fail(reason: str) -> "NoReturn":
    print(f"GREEN CHECK: FAIL ({reason})")
    raise _Stop()


def load_json(path: str):
    if not os.path.exists(path):
        fail(f"{path} not found")
    try:
        with open(path) as fh:
            return json.load(fh)
    except json.JSONDecodeError as exc:
        fail(f"{path} is not valid JSON: {exc}")


def main() -> None:
    b = load_json("baselines.json")

    # schema
    for key in ("model", "dtype", "ttft_s", "tpot_s", "batch"):
        if key not in b:
            fail(f"baselines.json missing key: {key}")
    if not isinstance(b["ttft_s"], dict) or not b["ttft_s"]:
        fail("ttft_s must be a non-empty object keyed by prompt length")
    if not isinstance(b["batch"], dict):
        fail("batch must be an object keyed by batch size")
    for size in ("1", "4", "8"):
        if size not in b["batch"]:
            fail(f"batch missing size {size}")

    # sanity 1: TTFT rises with prompt length - the day's actual physics.
    # Prefill reads the whole prompt before the first token, so a 2048-token
    # prompt must pay a visibly larger TTFT than a 128-token one (the reference
    # T4 run measured 0.037 s vs 0.312 s). A flat TTFT means prefill was not
    # measured (cached prompt, wrong timestamps, or a reused stream).
    tpot = b["tpot_s"]
    if not isinstance(tpot, (int, float)) or tpot <= 0:
        fail(f"tpot_s not a positive number: {tpot}")
    for plen, ttft in b["ttft_s"].items():
        if not isinstance(ttft, (int, float)) or ttft <= 0:
            fail(f"ttft_s[{plen}] not a positive number: {ttft}")
    if not b["ttft_s"]["2048"] > b["ttft_s"]["128"]:
        fail(f"TTFT did not rise with prompt length "
             f"(128: {b['ttft_s']['128']}, 2048: {b['ttft_s']['2048']}); "
             "prefill is not being measured")

    # sanity 2: batch-8 throughput beats batch-1
    b1, b8 = b["batch"]["1"], b["batch"]["8"]
    if not (isinstance(b1, (int, float)) and isinstance(b8, (int, float))):
        fail("batch tokens/s values must be numbers")
    if not b8 > b1:
        fail(f"batch-8 throughput ({b8}) not above batch-1 ({b1})")

    # sanity 3: measured KV within a factor of 2 of the formula
    kv = load_json("kv_check.json")
    measured = kv.get("measured_kb_per_token")
    if not isinstance(measured, (int, float)) or measured <= 0:
        fail("kv_check.json needs a positive measured_kb_per_token")
    lo, hi = KV_FORMULA_KB_PER_TOKEN / 2, KV_FORMULA_KB_PER_TOKEN * 2
    if not lo <= measured <= hi:
        fail(f"measured KV {measured} KB/token outside 2x of formula "
             f"{KV_FORMULA_KB_PER_TOKEN} (allowed {lo:.1f} to {hi:.1f})")

    print(f"ttft lengths: {sorted(b['ttft_s'])}, tpot_s: {tpot}")
    print(f"batch tokens/s 1/4/8: {b['batch']['1']}/{b['batch']['4']}/"
          f"{b['batch']['8']}")
    print(f"KV measured {measured} KB/token vs formula "
          f"{KV_FORMULA_KB_PER_TOKEN} KB/token")
    print("GREEN CHECK: PASS")


try:
    main()
except _Stop:
    # A notebook cell cannot exit nonzero without printing a red traceback over
    # the result line, so only signal by exit code when run as a plain script.
    try:
        get_ipython()  # defined only inside IPython/Colab
    except NameError:
        raise SystemExit(1)


ttft lengths: ['128', '2048', '512'], tpot_s: 0.044
batch tokens/s 1/4/8: 26.8/43.4/84.9
KV measured 28.0 KB/token vs formula 28.0 KB/token
GREEN CHECK: PASS
